# Data Pipelines for Language Model Pretraining

Before the model sees a single token, data has to be acquired, cleaned, tokenized, packed, and served to the training loop efficiently. A bad data pipeline is silent — it does not crash, it just trains a worse model, or trains a good model slowly, or silently duplicates examples in ways that look like fast convergence but are actually memorization.

This notebook builds the full pretraining data pipeline from scratch:

- Why `IterableDataset` exists and when you need it over `Dataset`
- Tokenizing on-the-fly vs pre-tokenized shards — the tradeoffs
- **Packing**: concatenating documents into fixed-length chunks with no padding waste — the pack-and-split loop, the boundary token, and the math behind why padding wastes 20–40% of effective training compute
- **Shuffle buffers**: why you cannot globally shuffle an `IterableDataset` and how a fixed-size buffer approximates it
- The `DataLoader` worker process model and why tokenization belongs in the worker, not the main process
- **Data deduplication**: why duplicates distort loss curves and make you think your model is better than it is, and a practical MinHash LSH implementation for near-duplicate detection
- A `PretrainingDataset` class that handles all of the above and plugs directly into the training loop

:::{.callout-note}
## References
The packing strategy and JSONL shard format follow the approach in [nanoGPT](https://github.com/karpathy/nanoGPT) — specifically the `data/` prepare scripts. The `DataLoader` design draws from [LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch) Chapter 2 dataloader. We assume the `Tokenizer` class built in [Notebook 02](/courses/llm/02-tokenization.html).

:::

## `Dataset` vs `IterableDataset`

PyTorch gives you two base classes for datasets:

**`Dataset`** requires implementing `__len__` and `__getitem__`. The `DataLoader` uses `__len__` to build an index array and shuffles it. This means the entire dataset index must fit in memory — fine for datasets up to a few hundred thousand examples, impractical for pretraining corpora that span terabytes.

**`IterableDataset`** requires only `__iter__`. There is no concept of length or random access. The `DataLoader` just calls `iter(dataset)` and pulls examples one at a time. [This is the only option]{.mark} when the dataset is too large to index in memory — which is always true for pretraining.

The cost: **no global shuffle**. You cannot randomly permute a stream of 100 billion tokens. Instead, we approximate shuffling with a buffer (Section 4).

In [ ]:
import torch
from torch.utils.data import IterableDataset, DataLoader
from pathlib import Path
import json

class DocumentDataset(IterableDataset):
    """
    Streams documents from a directory of JSONL files.
    Each line is a JSON object with a "text" field.

    This is the base — we will wrap it with packing and shuffling below.
    """

    def __init__(self, data_dir: str, split: str = 'train'):
        super().__init__()
        self.files = sorted(Path(data_dir).glob(f'{split}_*.jsonl'))
        if not self.files:
            raise FileNotFoundError(
                f"No files matching {split}_*.jsonl in {data_dir}"
            )
        print(f"DocumentDataset: {len(self.files)} files for split='{split}'")

    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()

        # If using DataLoader workers, each worker handles a subset of files
        # This prevents workers from reading the same file simultaneously
        files = self.files
        if worker_info is not None:
            files = [f for i, f in enumerate(files)
                     if i % worker_info.num_workers == worker_info.id]

        for path in files:
            with open(path, 'r', encoding='utf-8') as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        doc = json.loads(line)
                        yield doc['text']
                    except (json.JSONDecodeError, KeyError):
                        continue   # skip malformed lines silently

[The worker file-sharding pattern is critical.]{.underline} Without it, every worker reads every file and you get every document `num_workers` times — a silent duplication that makes loss appear to converge faster than it actually is.

## The Padding Problem

The naive approach to building fixed-length training batches is to tokenize each document independently and pad shorter documents to `max_seq_len`:

```
doc 1: [tok tok tok tok tok tok tok tok PAD PAD PAD PAD]  ← 8 real, 4 padded
doc 2: [tok tok PAD PAD PAD PAD PAD PAD PAD PAD PAD PAD]  ← 2 real, 10 padded
doc 3: [tok tok tok tok tok tok tok tok tok tok tok tok]  ← 12 real, 0 padded
```

The `ignore_index=PAD_ID` argument in `F.cross_entropy` means pad tokens contribute zero loss. But they still cost compute in the forward pass — every position, padded or not, runs through all the attention and FFN operations.

For a real corpus where document length follows a *power-law* distribution (many short documents, a few very long ones), the average padding fraction is typically [20–40%]{.mark}. You are paying for 20–40% of your GPU-hours to process tokens that teach the model nothing.

**The math:**

$$\text{effective throughput} = \text{raw throughput} \times (1 - \text{padding fraction})$$

[At 30% padding: a run that would take 100 GPU-hours of effective compute actually costs 143 GPU-hours.]{.mark} For a multi-week pretraining run, this is the difference between finishing and not finishing.

## Packing: The Fix

Packing eliminates padding by concatenating documents into a continuous stream of tokens and slicing it into fixed-length chunks:

```
doc 1 tokens:  [t t t t t t t t]
doc 2 tokens:  [t t]
doc 3 tokens:  [t t t t t t t t t t t t]
                              ↓ concatenate with EOS between documents
stream:        [t t t t t t t t EOS t t EOS t t t t t t t t t t t t]
                              ↓ slice into blocks of size block_size+1
block 1:       [t t t t t t t t EOS t t EOS t]
block 2:       [t t t t t t t t t t t]
```

[Zero padding, zero wasted compute.]{.mark} The `EOS` token at each document boundary teaches the model that context resets — it must not attend across document boundaries in a meaningful way. Placing `EOS` in the stream (rather than masking cross-document attention) is the standard pretraining approach because it is simpler and works in practice.

In [ ]:
from torch.utils.data import IterableDataset


class PackedDataset(IterableDataset):
    """
    Wraps a document stream, tokenizes each document, and packs tokens
    into fixed-length chunks with no padding.

    Yields:
        (x, y) pairs where x and y are LongTensors of shape (block_size,)
        and y = x shifted left by one (next-token prediction targets).
    """

    def __init__(
        self,
        doc_dataset: IterableDataset,
        tokenizer,
        block_size: int,
        eos_id: int = None,
    ):
        super().__init__()
        self.doc_dataset = doc_dataset
        self.tokenizer   = tokenizer
        self.block_size  = block_size
        self.eos_id      = eos_id if eos_id is not None else tokenizer.eos_id

    def __iter__(self):
        buffer = []   # rolling token buffer

        for doc in self.doc_dataset:
            # Tokenize document and append EOS
            tokens = self.tokenizer.encode(doc)
            if not tokens:
                continue
            buffer.extend(tokens)
            buffer.append(self.eos_id)

            # Yield complete chunks as soon as we have enough tokens
            # We need block_size + 1 tokens to make one (x, y) pair
            while len(buffer) >= self.block_size + 1:
                chunk  = buffer[:self.block_size + 1]
                buffer = buffer[self.block_size + 1:]

                x = torch.tensor(chunk[:-1], dtype=torch.long)
                y = torch.tensor(chunk[1:],  dtype=torch.long)
                yield x, y

        # Final partial chunk is discarded — the loss from one partial
        # chunk is negligible and avoids reintroducing padding

**Why `block_size + 1`?** To make a next-token prediction pair, we need `block_size` input tokens and `block_size` target tokens. The targets are the inputs shifted left by one: `x = tokens[:-1]`, `y = tokens[1:]`. So we need `block_size + 1` tokens to produce one pair with no overlap.[^blocksize]

**The buffer is a plain Python list.** We extend it with each document's tokens and pop chunks off the front. This is $O(n)$ in the number of tokens appended, but since we immediately pop chunks as soon as they're ready, the buffer never grows beyond `block_size + 1 + max_doc_length`. For typical corpora (max doc length ~10K tokens, block_size ~1K), the buffer stays small.

[^blocksize]: `block_size + 1` is not a magic number — it falls directly from the shift: to produce `x[0..T-1]` and `y[1..T]`, you need tokens `0..T`, which is `T+1 = block_size+1` tokens.

## Shuffle Buffers

Without shuffling, consecutive chunks come from consecutive documents. If your corpus has structure — all news articles together, then all books, then all code — the model will cycle through domains, see gradient signals from one domain at a time, and potentially overfit to the current domain before moving to the next. More subtly, the Adam optimizer's moment estimates will be tuned to the current domain statistics.

[A shuffle buffer maintains a fixed-size pool of examples.]{.underline} When you request an example, you pick one randomly from the pool and replace it with the next example from the stream:

```
Buffer (size 4):  [doc_A_chunk2, doc_C_chunk1, doc_B_chunk3, doc_A_chunk5]
Request example:  pick random index 2 → yield doc_B_chunk3
Refill:           read next chunk from stream → doc_D_chunk1
Buffer:           [doc_A_chunk2, doc_C_chunk1, doc_D_chunk1, doc_A_chunk5]
```

This is not a perfect shuffle (examples near the buffer boundary are correlated), but it is good enough for pretraining in practice.

In [ ]:
import random

class ShuffledDataset(IterableDataset):
    """
    Wraps any IterableDataset with reservoir-style shuffle buffering.

    buffer_size controls the shuffle quality:
    - buffer_size=1:      no shuffle (deterministic order)
    - buffer_size=1000:   light shuffle (adjacent examples within ~1000 window)
    - buffer_size=100000: strong shuffle (memory cost: ~100K examples)

    For pretraining on packed token chunks (~2KB each):
    - buffer_size=10000 uses ~20GB RAM — too large for most machines
    - buffer_size=1000  uses ~2GB RAM  — reasonable
    - buffer_size=500   uses ~1GB RAM  — good default for single-GPU setups
    """

    def __init__(
        self,
        dataset: IterableDataset,
        buffer_size: int = 500,
        seed: int = 42,
    ):
        super().__init__()
        self.dataset     = dataset
        self.buffer_size = buffer_size
        self.seed        = seed

    def __iter__(self):
        rng    = random.Random(self.seed)
        buffer = []

        iterator = iter(self.dataset)

        # Fill the buffer initially
        try:
            while len(buffer) < self.buffer_size:
                buffer.append(next(iterator))
        except StopIteration:
            # Dataset is smaller than buffer — shuffle what we have
            rng.shuffle(buffer)
            yield from buffer
            return

        # Stream: for each new example, yield a random one from the buffer
        for item in iterator:
            idx = rng.randrange(len(buffer))
            yield buffer[idx]
            buffer[idx] = item

        # Drain remaining buffer in random order
        rng.shuffle(buffer)
        yield from buffer

**Buffer size guidance.** Each packed chunk is `block_size × 4` bytes (int32). For `block_size=256`: each chunk is ~1KB. For `block_size=1024`: each chunk is ~4KB.

In [ ]:
def buffer_memory_mb(buffer_size: int, block_size: int) -> float:
    bytes_per_chunk = block_size * 4 * 2   # x and y tensors, int32
    return buffer_size * bytes_per_chunk / 1e6

print(buffer_memory_mb(500,   256))    # 1.0 MB   — trivial
print(buffer_memory_mb(500,  1024))    # 4.1 MB   — trivial
print(buffer_memory_mb(5000, 1024))    # 40.9 MB  — fine
print(buffer_memory_mb(50000, 1024))   # 409.6 MB — getting large

For the nano model with `block_size=256`, `buffer_size=1000` is a good default — 2MB of memory, decent shuffle quality.

## The `DataLoader` Worker Model

`DataLoader` with `num_workers > 0` spawns separate Python processes to prefetch batches while the GPU runs the forward pass. Each worker runs `__iter__` independently. This is why the file-sharding in `DocumentDataset` is necessary — each worker must read a disjoint subset of files.

In [ ]:
def make_dataloader(
    dataset: IterableDataset,
    batch_size: int,
    num_workers: int = 2,
) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=True,    # <1>
        prefetch_factor=2,  # <2>
    )

1. Allocates batch tensors in page-locked (pinned) host memory. The GPU can DMA-transfer from pinned memory without the CPU copying to a staging buffer first. For small models where the data transfer is a bottleneck, this can improve throughput by 10–20%.
2. Each worker keeps 2 batches ready in a queue. With 4 workers and `prefetch_factor=2`, you have up to 8 batches prefetched — enough to keep the GPU busy through any tokenization jitter.

**Why tokenization belongs in the worker:**

In [ ]:
# WRONG: tokenize in __init__ or outside the dataset
# Main process does all tokenization — workers do nothing useful
all_tokens = tokenizer.encode(text)   # blocks main process for minutes

# CORRECT: tokenize in __iter__ inside the dataset
# Each worker tokenizes its own shard independently (see PackedDataset above)
# tokens = self.tokenizer.encode(doc)  — called inside __iter__

[When tokenization is in `__iter__`, each worker tokenizes its shard in parallel with the others and with GPU computation.]{.mark} When tokenization is in the main process, you serialize the bottleneck.

**The seed problem with multiple workers.** Each worker process gets the same random seed by default. If you use `random.Random(seed)` in `ShuffledDataset.__iter__`, all workers will produce the same shuffle order — defeating the purpose. Fix:

In [ ]:
def __iter__(self):
    worker_info = torch.utils.data.get_worker_info()
    seed = self.seed
    if worker_info is not None:
        seed = self.seed + worker_info.id   # different seed per worker
    rng = random.Random(seed)
    ...

## Pre-tokenized Shards

For large-scale training (tens of billions of tokens), tokenizing on-the-fly in the worker adds latency that can bottleneck the pipeline even with many workers. The solution: pre-tokenize the corpus once and save it as binary shards of token IDs.

In [ ]:
import numpy as np
from pathlib import Path

def pretokenize_corpus(
    input_dir: str,
    output_dir: str,
    tokenizer,
    shard_size: int = 100_000_000,   # 100M tokens per shard
    val_fraction: float = 0.05,
):
    """
    Tokenize all JSONL files in input_dir and save as .npy shards.

    Each shard is a numpy array of uint16 token IDs (vocab_size <= 65535).  # <1>
    Output structure:
        output_dir/
            train_000.npy   (100M tokens, ~200MB as uint16)
            train_001.npy
            ...
            val_000.npy
    """
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    all_tokens: list[int] = []
    shard_idx  = 0
    n_shards   = 0   # approximate — used to compute train/val split

    def flush_shard(tokens: list[int], shard_idx: int):
        shard = np.array(tokens, dtype=np.uint16)
        # First (1 - val_fraction) of shards go to train
        split = 'val' if shard_idx == 0 else 'train'  # first shard → val
        fname = f"{output_dir}/{split}_{shard_idx:03d}.npy"
        np.save(fname, shard)
        print(f"  Saved {fname}  ({len(shard):,} tokens)")

    for path in sorted(Path(input_dir).glob('*.jsonl')):
        with open(path, encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    text   = json.loads(line)['text']
                    tokens = tokenizer.encode(text)
                    tokens.append(tokenizer.eos_id)
                    all_tokens.extend(tokens)
                except Exception:
                    continue

                while len(all_tokens) >= shard_size:
                    flush_shard(all_tokens[:shard_size], shard_idx)
                    all_tokens = all_tokens[shard_size:]
                    shard_idx += 1

    if all_tokens:
        flush_shard(all_tokens, shard_idx)

    print(f"Done. {shard_idx + 1} shards written.")

1. `uint16` halves the disk space vs `int32` with no loss for `vocab_size ≤ 65535`. Our nano tokenizer has `vocab_size=4096`, well under the `uint16` maximum. The `.astype(np.int64)` on load converts back to the dtype PyTorch's embedding layer expects.

In [ ]:
class ShardedTokenDataset(IterableDataset):
    """
    Reads pre-tokenized .npy shards and yields (x, y) pairs.
    Worker-aware: each worker reads a disjoint subset of shards.
    """

    def __init__(self, shard_dir: str, split: str, block_size: int):
        super().__init__()
        self.shards     = sorted(Path(shard_dir).glob(f'{split}_*.npy'))
        self.block_size = block_size
        if not self.shards:
            raise FileNotFoundError(f"No {split}_*.npy shards in {shard_dir}")

    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()
        shards = self.shards
        if worker_info is not None:
            shards = [s for i, s in enumerate(shards)
                      if i % worker_info.num_workers == worker_info.id]

        for shard_path in shards:
            tokens = np.load(shard_path).astype(np.int64)  # uint16 → int64
            T = len(tokens)
            for start in range(0, T - self.block_size, self.block_size + 1):
                chunk = tokens[start : start + self.block_size + 1]
                x = torch.from_numpy(chunk[:-1])
                y = torch.from_numpy(chunk[1:])
                yield x, y

## Data Deduplication

[Near-duplicate documents in a training corpus produce two problems:]{.underline}

**Problem 1 — Loss distortion.** If document D appears 10 times in your corpus, the model sees it 10 times per epoch. The loss on D decreases faster than on unique documents, making overall train loss look better than it is. You think you have a good model; you have a model that has memorized 10 copies of D.

**Problem 2 — Memorization.** Duplicated documents are memorized disproportionately. A model trained on a corpus with 1% of documents appearing 10+ times will verbatim-reproduce those documents when prompted with their first few tokens — a privacy and quality problem.

The Pile, C4, and RedPajama all include deduplication as a preprocessing step. Here is a practical implementation using MinHash LSH (Locality Sensitive Hashing):

In [ ]:
import hashlib
from collections import defaultdict

class MinHashDeduplicator:
    """
    Near-duplicate detection using MinHash LSH.

    MinHash approximates the Jaccard similarity between documents:
        J(A, B) = |A ∩ B| / |A ∪ B|

    where A and B are sets of n-grams. Two documents with J > threshold
    are considered near-duplicates — keep one, discard the rest.
    """

    def __init__(
        self,
        n_hashes:   int   = 128,
        threshold:  float = 0.8,
        ngram_size: int   = 5,
    ):
        self.n_hashes   = n_hashes
        self.threshold  = threshold
        self.ngram_size = ngram_size

        # Precompute hash function parameters: h(x) = (ax + b) mod p
        rng = np.random.RandomState(42)
        p   = (1 << 31) - 1   # Mersenne prime
        self._a = rng.randint(1, p, size=n_hashes, dtype=np.int64)
        self._b = rng.randint(0, p, size=n_hashes, dtype=np.int64)
        self._p = p

        # LSH bands for fast candidate retrieval
        # b bands × r rows = n_hashes; threshold ≈ (1/b)^(1/r)
        # For threshold=0.8, n_hashes=128: b=16, r=8 → threshold≈0.8
        self._n_bands = 16
        self._n_rows  = n_hashes // self._n_bands

        # Inverted index: band_hash → list of doc_ids
        self._buckets: dict[int, list[int]] = defaultdict(list)
        self._signatures: list[np.ndarray] = []
        self._is_duplicate: list[bool]     = []

    def _shingles(self, text: str) -> set[int]:
        """Convert text to a set of hashed character n-grams."""
        text    = text.lower()
        shingles = set()
        for i in range(len(text) - self.ngram_size + 1):
            ngram = text[i : i + self.ngram_size].encode('utf-8')
            h     = int(hashlib.md5(ngram).hexdigest(), 16) % self._p
            shingles.add(h)
        return shingles

    def _minhash(self, shingles: set[int]) -> np.ndarray:
        """Compute the MinHash signature (k-dimensional vector of min hashes)."""
        s = np.array(list(shingles), dtype=np.int64)
        # Vectorized: compute h(x) = (ax + b) % p for all hash functions at once
        hashes = (self._a[:, None] * s[None, :] + self._b[:, None]) % self._p
        return hashes.min(axis=1)  # (n_hashes,)

    def is_duplicate(self, text: str) -> bool:
        """Returns True if text is a near-duplicate of a previously seen document."""
        doc_id    = len(self._signatures)
        shingles  = self._shingles(text)
        signature = self._minhash(shingles)
        self._signatures.append(signature)

        # Check candidate duplicates via LSH bands
        candidates = set()
        for band in range(self._n_bands):
            start     = band * self._n_rows
            end       = start + self._n_rows
            band_sig  = tuple(signature[start:end].tolist())
            band_hash = hash(band_sig)

            for cand_id in self._buckets[band_hash]:
                candidates.add(cand_id)
            self._buckets[band_hash].append(doc_id)

        # Verify candidates with exact Jaccard estimate
        for cand_id in candidates:
            jaccard = np.mean(signature == self._signatures[cand_id])
            if jaccard >= self.threshold:
                self._is_duplicate.append(True)
                return True

        self._is_duplicate.append(False)
        return False

**The MinHash math in brief.** For a hash function $h(x) = (ax + b) \mod p$ drawn uniformly at random, and two sets $A$, $B$ of shingles:

$$P\left(\min_{x \in A} h(x) = \min_{x \in B} h(x)\right) = \frac{|A \cap B|}{|A \cup B|} = J(A, B)$$

The minimum hash value is equally likely to come from either set only when the element that achieves the minimum is shared. With $k$ independent hash functions, the fraction of positions where both signatures agree is an unbiased estimate of $J(A, B)$ with variance $\frac{J(1-J)}{k}$. With $k=128$, the standard deviation is $\sqrt{J(1-J)/128} \leq 0.044$ — accurate enough for deduplication.

## The Complete `PretrainingDataset`

Putting it all together — a single class that handles document streaming, packing, shuffling, and worker sharding:

In [ ]:
class PretrainingDataset(IterableDataset):
    """
    Complete pretraining data pipeline.

    Supports two modes:
    - On-the-fly: reads JSONL files and tokenizes in the worker
    - Sharded:    reads pre-tokenized .npy shards (faster for large corpora)

    Usage (on-the-fly):
        dataset = PretrainingDataset.from_jsonl(
            data_dir='data/raw',
            tokenizer=tok,
            block_size=256,
            split='train',
            buffer_size=1000,
        )

    Usage (sharded):
        dataset = PretrainingDataset.from_shards(
            shard_dir='data/tokenized',
            block_size=256,
            split='train',
            buffer_size=1000,
        )
    """

    def __init__(
        self,
        source: IterableDataset,
        buffer_size: int = 1000,
        seed: int = 42,
    ):
        super().__init__()
        self.source      = source
        self.buffer_size = buffer_size
        self.seed        = seed

    @classmethod
    def from_jsonl(
        cls,
        data_dir:    str,
        tokenizer,
        block_size:  int,
        split:       str   = 'train',
        buffer_size: int   = 1000,
        seed:        int   = 42,
    ):
        docs   = DocumentDataset(data_dir, split=split)
        packed = PackedDataset(docs, tokenizer, block_size)
        return cls(packed, buffer_size=buffer_size, seed=seed)

    @classmethod
    def from_shards(
        cls,
        shard_dir:   str,
        block_size:  int,
        split:       str = 'train',
        buffer_size: int = 1000,
        seed:        int = 42,
    ):
        shards = ShardedTokenDataset(shard_dir, split, block_size)
        return cls(shards, buffer_size=buffer_size, seed=seed)

    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()
        seed = self.seed + (worker_info.id if worker_info is not None else 0)

        rng    = random.Random(seed)
        buffer = []
        iterator = iter(self.source)

        # Fill buffer
        try:
            while len(buffer) < self.buffer_size:
                buffer.append(next(iterator))
        except StopIteration:
            rng.shuffle(buffer)
            yield from buffer
            return

        for item in iterator:
            idx = rng.randrange(len(buffer))
            yield buffer[idx]
            buffer[idx] = item

        rng.shuffle(buffer)
        yield from buffer

## Building the TinyShakespeare Pipeline

For the nano model trained throughout this series, here is the complete pipeline setup:

In [ ]:
import urllib.request

# Download dataset
url  = ("https://raw.githubusercontent.com/karpathy/char-rnn"
        "/master/data/tinyshakespeare/input.txt")
text = urllib.request.urlopen(url).read().decode('utf-8')

# Write as JSONL split into train/val
Path('data/shakespeare').mkdir(parents=True, exist_ok=True)
split_point = int(0.9 * len(text))

with open('data/shakespeare/train_000.jsonl', 'w') as f:
    chunk_size = 500
    for i in range(0, split_point, chunk_size):
        chunk = text[i : i + chunk_size].strip()
        if chunk:
            f.write(json.dumps({'text': chunk}) + '\n')

with open('data/shakespeare/val_000.jsonl', 'w') as f:
    for i in range(split_point, len(text), chunk_size):
        chunk = text[i : i + chunk_size].strip()
        if chunk:
            f.write(json.dumps({'text': chunk}) + '\n')

# Load tokenizer (trained in Notebook 02)
from pathlib import Path
tok = Tokenizer.load('nano_tokenizer.json')

# Build datasets
train_dataset = PretrainingDataset.from_jsonl(
    data_dir='data/shakespeare',
    tokenizer=tok,
    block_size=256,
    split='train',
    buffer_size=500,
)
val_dataset = PretrainingDataset.from_jsonl(
    data_dir='data/shakespeare',
    tokenizer=tok,
    block_size=256,
    split='val',
    buffer_size=100,
)

# Build dataloaders
train_loader = make_dataloader(train_dataset, batch_size=8, num_workers=2)
val_loader   = make_dataloader(val_dataset,   batch_size=8, num_workers=1)

# Verify pipeline
x, y = next(iter(train_loader))
print(f"Batch shape: x={x.shape}, y={y.shape}")   # (8, 256), (8, 256)
print(f"No padding: min token id = {x.min().item()}")
print(f"Sample tokens: {x[0, :20].tolist()}")
print(f"Decoded: '{tok.decode(x[0, :20].tolist())}'")

### Pipeline statistics

Before training, measure your pipeline's throughput to make sure data loading is not the bottleneck:

In [ ]:
import time

def benchmark_dataloader(loader, n_batches=100):
    """Measure tokens/sec throughput of the data pipeline."""
    iterator  = iter(loader)
    t0        = time.time()
    total_tok = 0

    for _ in range(n_batches):
        x, y      = next(iterator)
        total_tok += x.numel()

    elapsed = time.time() - t0
    tps     = total_tok / elapsed
    print(f"Data pipeline throughput: {tps:,.0f} tokens/sec")
    print(f"  ({n_batches} batches × {x.shape} = {total_tok:,} tokens in {elapsed:.2f}s)")
    return tps

# Rule of thumb: data pipeline should be at least 2× faster than
# the training step (GPU computation). If not, increase num_workers
# or switch to pre-tokenized shards.
data_tps = benchmark_dataloader(train_loader)

## Summary

| Concept | Key detail |
|---|---|
| `IterableDataset` | No `__len__`, no random access. Required for corpora that don't fit in memory. |
| Worker file-sharding | Each worker reads disjoint file subset — prevents silent $N\times$ duplication. |
| Padding waste | Average 20–40% for power-law document lengths — pays full GPU cost for zero signal. |
| Packing | Concatenate docs + EOS, slice into `block_size+1` chunks. Zero padding. |
| `block_size + 1` | Need one extra token to form $(x, y)$ pairs via left-shift. |
| Shuffle buffer | Fixed-size pool, reservoir sampling. Buffer of 1000 costs ~4MB for `block_size=1024`. |
| Worker seed offset | `seed + worker_info.id` — different shuffle per worker. |
| `pin_memory=True` | Allocates tensors in page-locked memory for faster GPU DMA transfer. |
| Tokenize in worker | Parallelizes tokenization across workers and GPU computation. |
| `uint16` shards | Half the disk/memory of `int32`. Valid for `vocab_size ≤ 65535`. |
| MinHash LSH | Estimates Jaccard similarity in $O(k \cdot |\text{shingles}|)$. With $k=128$: std $\leq 0.044$. |
| Deduplication | Prevents loss distortion and memorization from repeated documents. |
| Pipeline benchmark | Data tps should be $\geq 2\times$ training tps — otherwise data loading is the bottleneck. |

: {tbl-colwidths="[35,65]"}

## Exercises

**1.** Run `benchmark_dataloader` on the TinyShakespeare pipeline with `num_workers` set to 0, 1, 2, and 4. Plot throughput vs `num_workers`. Identify the point of diminishing returns for your machine.

**2.** Modify `PackedDataset` to also yield the attention mask — a tensor of ones for all positions (since there is no padding). Then modify it to support a `max_doc_length` argument: documents longer than this are truncated rather than split across chunk boundaries.

**3.** Implement the `pretokenize_corpus` function and run it on TinyShakespeare. Then build a `ShardedTokenDataset` from the output and verify it produces identical batches to the on-the-fly pipeline (same seed, same block_size).

**4.** Run `MinHashDeduplicator` on the TinyShakespeare JSONL files. What fraction of chunks are near-duplicates at threshold=0.8? At threshold=0.5? Plot the duplicate fraction vs threshold.

**5.** The current `ShuffledDataset` has an edge case: if the dataset is exhausted before the buffer is full, it yields a globally shuffled buffer. This is correct, but the shuffled order is fixed to the initial seed. Add an `epoch` parameter so repeated passes over a small dataset produce different orderings.

**6.** Extend `DocumentDataset` to support streaming from HuggingFace `datasets` (the `load_dataset` API with `streaming=True`). Wrap it so that the `text` field is extracted from whatever column name is provided as an argument.

■